# Tools: Giving LLM Agents the Ability to Act

An LLM on its own can only generate text. **Tools** let an agent look things up, run code, and call external services — the model decides *when* and *how* to call a tool, and your code executes it.

This notebook walks through five tool patterns, mostly using [`smolagents`](https://github.com/huggingface/smolagents) (with `LiteLLMModel` as a thin, provider-agnostic wrapper around `gpt-4o-mini`), plus one example using the native OpenAI Agents SDK:

1. **Custom Database Tool** — a `@tool`-decorated function that looks up order status.
2. **Python Interpreter Tool** — letting the agent execute Python to solve a computation.
3. **Search + Calculator** — combining a web search tool with the agent's own reasoning/math.
4. **Stock Price Tool** — a `function_tool` wired into the OpenAI Agents SDK, backed by `yfinance`.
5. **Web Browser Tool** — an agent that writes its own scraping code against a live webpage.

---
## Step 1: Install Dependencies

In [4]:
!pip install -q smolagents litellm duckduckgo-search yfinance openai-agents python-dotenv beautifulsoup4 httpx


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


---
## Step 2: Set Up Environment Variables
All five demos call `gpt-4o-mini` under the hood, so we only need an OpenAI API key.

In [5]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

if "OPENAI_API_KEY" not in os.environ or not os.environ["OPENAI_API_KEY"]:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

---
## Tool 1: Custom Database Lookup
We define a plain Python function, decorate it with `@tool`, and hand it to a `CodeAgent`. The docstring's `Args` section is what the model reads to know how to call it — so it doubles as the tool's API contract.

> The original version of this demo pointed at a local Ollama model (`ollama/gemma3:4b` via `api_base="http://localhost:11434"`). Here we swap that for the hosted `gpt-4o-mini` so the notebook runs end-to-end in Colab with no local server.

In [6]:
from smolagents import CodeAgent, LiteLLMModel, tool

model = LiteLLMModel(model_id="gpt-4o-mini")

@tool
def get_order_status(order_id: str) -> str:
    """
    Retrieves the real-time status and estimated delivery for a customer order.

    Args:
        order_id: The unique alphanumeric order tracking ID (e.g., 'ORD-123').
    """
    # Simulated Database Lookup
    mock_db = {
        "ORD-101": {"status": "In Transit", "location": "Bengaluru Hub", "eta": "2 Hours"},
        "ORD-202": {"status": "Delivered", "location": "Mumbai", "eta": "N/A"},
        "ORD-303": {"status": "Processing", "location": "Warehouse", "eta": "2 Days"}
    }

    order = mock_db.get(order_id.upper())
    if order:
        return f"Order {order_id} is currently '{order['status']}' at {order['location']}. Estimated arrival: {order['eta']}."
    return f"Order ID {order_id} not found. Please verify the ID and try again."

agent = CodeAgent(
    tools=[get_order_status],
    model=model,
    add_base_tools=False  # We only want it to use our specific support tool
)

customer_query = "Hey, I've been waiting for my package ORD-101. Can you tell me where it is and when it will get here?"
agent.run(customer_query)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Hey, I've been waiting for my package ORD-101. Can you tell me where it is and when it will get here?           │
│                                                                                                                 │
╰─ LiteLLMModel - gpt-4o-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  order_status = get_order_status(order_id="ORD-101")                                                              
  print(order_status)                                                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Order ORD-101 is currently 'In Transit' at Bengaluru Hub. Estimated arrival: 2 Hours.

Out: None

[Step 1: Duration 1.55 seconds| Input tokens: 2,048 | Output tokens: 66]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Order ORD-101 is currently 'In Transit' at Bengaluru Hub. Estimated arrival: 2 hours.")            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Order ORD-101 is currently 'In Transit' at Bengaluru Hub. Estimated arrival: 2 hours.

[Step 2: Duration 1.51 seconds| Input tokens: 4,257 | Output tokens: 150]

"Order ORD-101 is currently 'In Transit' at Bengaluru Hub. Estimated arrival: 2 hours."

---
## Tool 2: Python Interpreter
`PythonInterpreterTool` gives the agent a sandboxed Python runtime it can call to compute exact answers instead of guessing numerically.

In [7]:
from smolagents import ToolCallingAgent, PythonInterpreterTool, LiteLLMModel

model = LiteLLMModel(model_id="gpt-4o-mini")

# Create a traditional JSON-based tool-calling agent with access to Python code execution
agent = ToolCallingAgent(
    tools=[PythonInterpreterTool()],
    model=model
)

result = agent.run("What is the 50th Fibonacci number? Compute it using Python.")
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the 50th Fibonacci number? Compute it using Python.                                                     │
│                                                                                                                 │
╰─ LiteLLMModel - gpt-4o-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'python_interpreter' with arguments: {'code': 'def fibonacci(n):\n    a, b = 0, 1\n    for _ in   │
│ range(n):\n        a, b = b, a + b\n    return a\n\nfibonacci(50)'}                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Stdout:

Output: 12586269025

[Step 1: Duration 2.83 seconds| Input tokens: 1,081 | Output tokens: 61]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': '12586269025'}                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: 12586269025

Final answer: 12586269025

[Step 2: Duration 1.44 seconds| Input tokens: 2,282 | Output tokens: 78]

12586269025


---
## Tool 3: Search + Calculator
`DuckDuckGoSearchTool` gives the agent live web access. `CodeAgent` can then combine that retrieved fact with its own arithmetic — no separate calculator tool is needed since the agent writes and runs Python directly.

In [8]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, LiteLLMModel

model = LiteLLMModel(model_id="gpt-4o-mini")

agent = CodeAgent(
    tools=[DuckDuckGoSearchTool()],
    model=model
)

result = agent.run(
    "Search for the distance between Earth and Mars today, "
    "and calculate how long light takes to travel that distance in minutes."
)
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Search for the distance between Earth and Mars today, and calculate how long light takes to travel that         │
│ distance in minutes.                                                                                            │
│                                                                                                                 │
╰─ LiteLLMModel - gpt-4o-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  distance_info = web_search(query="current distance between Earth and Mars")                                      
  print("Distance info:", distance_info)                                                                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Distance info: ## Search Results

[Solar System - Wikipedia](https://en.wikipedia.org/wiki/Solar_System)
The Solar System is an isolated single-star planetary system within the Milky Way Galaxy. The system formed about 
4.6 billion years ago when a dense region of a molecular cloud collapsed, creating the Sun and a protoplanetary 
disc from which the orbiting bodies assembled.

[Mars: Complete Information & Live Data | TheSkyLive](https://theskylive.com/mars-info)
The distance of Mars from Earth is currently 242,852,812.0 kilometers, equivalent to 1.6233707800 Astronomical 
Units. Light takes 13m 30.070s to travel from Mars and arrive to us.

[How far away is Mars - StarLust](https://starlust.org/how-far-away-is-mars-now/)
9 hours ago · The average distance between Earth and Mars is 140 million miles (225 million km), so based on that 
distance, I have calculated the time it would take to reach the red planet at the speed of different means of 
transportation.

[Distance Earth to Mars: Current and Historical 
Data](https://completeera.com/distance-earth-to-mars-current-and-historical-data/)
TL;DR: The distance between Earth and Mars varies between **33.9 million miles (54.6 million km)** at closest 
approach (opposition) and **250 million miles (401 million km)** at farthest (conjunction). Right now, Mars is 
miles away—check the live tracker below!

[How Far Away Is Mars from Earth? - TheSkyLive](https://theskylive.com/how-far-is-mars)
The distance of Mars from Earth is currently 243,606,336.47 Kilometers (151,402,322.23 Miles), equivalent to 
1.6284077797 Astronomical Units. Light takes 13 minutes and 32.583 seconds to travel from Mars and arrive to us.

[Distance From Earth to Mars: Current and Historical 
Data](https://completeera.com/distance-from-earth-to-mars-current-and-historical-data/)
TL;DR: The distance between Earth and Mars varies between **33.9 million miles (54.6 million km)** at its closest 
(opposition) and **250 million miles (401 million km)** at its farthest (conjunction). Right now, Mars is about 
miles away—check the live tracker below!

[How Far Is Mars from Earth Right Now? - SnapDistance](https://snapdistance.com/distance-to-mars)
Jul 8, 2026 · Right now, it's the number in the live widget below — anywhere from 54.6 million kilometres at 
closest approach to about 401 million kilometres when Mars is behind the Sun. Radio signals take between 3 and 22 
minutes to travel one way, which is why rovers on Mars don't get driven with a joystick.

[Distance, Brightness, and Apparent Size of Planets](https://www.timeanddate.com/astronomy/planets/distance)
Explore the distances between planets, Earth, and the Sun, plus their brightness and apparent size in the sky at 
any given time.

[How Long Would It Take to Get to Mars? | Mars from Earth, 
Speed...](https://www.britannica.com/science/How-Long-Would-It-Take-to-Get-to-Mars)
Distance to Mars. Earth is never still. It spins on its axis and orbits the Sun. Mars does the same.The closest 
distance between Mars and Earth is about 56 million km (35 million miles).

[Distance to Mars](https://distancetomars.com/)
Go To Mars. You're currently travelling at 7,000 pixels a second.At the current state of space technology, it will 
take at least 150 days to get to Mars. When we launched this website in 2013, the timeline for a manned mission to 
Mars was the 2030s.

Out: None

[Step 1: Duration 6.09 seconds| Input tokens: 2,043 | Output tokens: 155]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(time_minutes)                                                                                       
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 13.501183709594208

[Step 3: Duration 1.05 seconds| Input tokens: 8,650 | Output tokens: 442]

13.501183709594208


---
## Tool 4: Stock Price Lookup (OpenAI Agents SDK)
This example switches frameworks: instead of `smolagents`, we use OpenAI's native **Agents SDK**, which packages instructions, models, and tools directly into an `Agent` object. `@function_tool` turns any typed Python function into a callable tool, and `Runner` drives the LLM → tool call → result → LLM loop automatically.

In [9]:
import yfinance as yf
from agents import Agent, Runner, function_tool

@function_tool
def get_stock_price(ticker: str) -> str:
    info = yf.Ticker(ticker).fast_info
    return f"The price of {ticker} is {info.last_price:.2f} {info.currency}."

finance_agent = Agent(
    name="Finance Agent",
    instructions="You are a financial assistant. Use tools to look up prices.",
    tools=[get_stock_price],
    model="gpt-4o",
)

result = await Runner.run(finance_agent, "What is the stock price of AAPL?")
print(result.final_output)

The current stock price of AAPL is $333.63 USD.


---
## Tool 5: Web Browser
Earlier iterations of this demo let the agent write its own scraping and DOM-selection code from scratch. That turned out to be unreliable: the model guessed CSS selectors from its training memory (e.g. Hacker News's old `storylink` class, retired years ago) instead of the site's current markup, and the scrape failed.

Here we fix that by making the scraping logic a **deterministic tool** — `scrape_hacker_news` — instead of freeform agent-written code. The agent's job shrinks to calling the tool and formatting the result; the parsing itself no longer depends on the model guessing correctly.

In [10]:
from smolagents import CodeAgent, LiteLLMModel, tool
import httpx
import json
from bs4 import BeautifulSoup

model = LiteLLMModel(model_id="gpt-4o-mini")

@tool
def scrape_hacker_news(top_n: int) -> str:
    """
    Fetches the Hacker News front page and returns the top stories' titles and point counts as JSON.

    Args:
        top_n: Number of top stories to return.
    """
    response = httpx.get("https://news.ycombinator.com/", timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    stories = []
    for row in soup.find_all("tr", class_="athing")[:top_n]:
        title_el = row.select_one(".titleline > a")
        title = title_el.get_text() if title_el else "Unknown"

        subtext_row = row.find_next_sibling("tr")
        score_el = subtext_row.find(class_="score") if subtext_row else None
        points = score_el.get_text().split()[0] if score_el else "0"

        stories.append({"title": title, "points": points})

    return json.dumps(stories)

agent = CodeAgent(
    tools=[scrape_hacker_news],
    model=model,
    add_base_tools=False,
)

result = agent.run("Get the top 5 Hacker News stories with their titles and point counts.")
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Get the top 5 Hacker News stories with their titles and point counts.                                           │
│                                                                                                                 │
╰─ LiteLLMModel - gpt-4o-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  top_stories = scrape_hacker_news(top_n=5)                                                                        
  print(top_stories)                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
[{"title": "Sharing AI progress in mathematics", "points": "919"}, {"title": "Reasons to Dislike AI Coding", 
"points": "15"}, {"title": "A font recreated from photographs of classic Commodore 64 keycaps", "points": "7"}, 
{"title": "Strands Decider 2B: a small, open-source, decision model", "points": "175"}, {"title": "Decisions API is
in public beta", "points": "296"}]

Out: None

[Step 1: Duration 2.59 seconds| Input tokens: 2,035 | Output tokens: 88]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(top_stories)                                                                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: [{"title": "Sharing AI progress in mathematics", "points": "919"}, {"title": "Reasons to Dislike AI 
Coding", "points": "15"}, {"title": "A font recreated from photographs of classic Commodore 64 keycaps", "points": 
"7"}, {"title": "Strands Decider 2B: a small, open-source, decision model", "points": "175"}, {"title": "Decisions 
API is in public beta", "points": "296"}]

[Step 2: Duration 1.55 seconds| Input tokens: 4,336 | Output tokens: 139]

[{"title": "Sharing AI progress in mathematics", "points": "919"}, {"title": "Reasons to Dislike AI Coding", "points": "15"}, {"title": "A font recreated from photographs of classic Commodore 64 keycaps", "points": "7"}, {"title": "Strands Decider 2B: a small, open-source, decision model", "points": "175"}, {"title": "Decisions API is in public beta", "points": "296"}]
